# Hafta 9 · Grover Arama Algoritması
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~50 dk · **Ortam:** Google Colab

Bu hafta sırasız aramayı karesel olarak hızlandıran **Grover algoritmasını** sıfırdan kuruyoruz. Önce NumPy ile genliklerin nasıl büyüdüğünü görüyoruz (oracle = işaret çevir, difüzyon = ortalama etrafında yansıt), sonra aynı şeyi Qiskit devresi olarak yazıp `Statevector` ile adım adım izliyoruz. Son olarak küçük bir **SAT problemi** ve **şifre bulma** örneği çözüyoruz.

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum ve yardımcı fonksiyonlar | 3 dk |
| A | Problem: kontrol fonksiyonu ve klasik arama | 4 dk |
| B | NumPy ile Grover: oracle ve ortalama etrafında yansıtma | 8 dk |
| C | Qiskit: faz oracle ve difüzyon devreleri | 8 dk |
| D | Genel `grover(n, marked)` ve Statevector ile adım adım izleme | 8 dk |
| E | Başarı olasılığı – iterasyon grafiği, aşma (overshoot) | 5 dk |
| F | Birden fazla çözüm (M > 1) | 3 dk |
| G | Uygulama 1: 3 değişkenli SAT (yardımcı kübitler, uncompute) | 7 dk |
| H | Uygulama 2: 4 bitlik şifre bulma ve listede arama | 4 dk |
| I | Gerçekçi değerlendirme: gürültü | ödev |
| J | Alıştırmalar (8 adet, `assert` ile) | ödev |

**Bit sırası:** Dersin tamamında olduğu gibi Qiskit sırası (q₀ en sağda). `'101'` → q₂ = 1, q₁ = 0, q₀ = 1, indeks 5.

## 0 · Kurulum

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import ZGate
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6"
aer = AerSimulator(seed_simulator=9)

def run_counts(qc, shots=2000):
    """Devreyi Aer simülatöründe çalıştırıp sayımları döndürür."""
    return aer.run(transpile(qc, aer, seed_transpiler=1), shots=shots).result().get_counts()

def plot_amps(a, marked=(), title="", ax=None):
    """Genlik çubuk grafiği: negatifler turuncu, hedefler lacivert, kesikli çizgi = ortalama."""
    a = np.real_if_close(np.asarray(a)).real
    N = len(a); n = int(np.log2(N)); show = ax is None
    if ax is None: fig, ax = plt.subplots(figsize=(max(5, 0.55*N), 3))
    cols = [ORANGE if v < -1e-12 else (NAVY if i in marked else BLUE) for i, v in enumerate(a)]
    ax.bar(range(N), a, color=cols, width=0.62); ax.axhline(0, color=GRAY, lw=0.8)
    ax.axhline(a.mean(), color=NAVY, ls="--", lw=1.2, label=f"ortalama = {a.mean():.3f}")
    ax.set_xticks(range(N)); ax.set_xticklabels([format(i, f"0{n}b") for i in range(N)], rotation=90 if N > 16 else 0, fontsize=8 if N > 8 else 10)
    ax.set_ylim(-1, 1.05); ax.set_title(title, color=NAVY, fontsize=11); ax.legend(frameon=False, fontsize=8, loc="lower right")
    if show: plt.show()

def plot_counts(counts, n, highlight=(), title=""):
    keys = [format(i, f"0{n}b") for i in range(2**n)]
    vals = [counts.get(k, 0) for k in keys]
    plt.figure(figsize=(max(5, 0.5*len(keys)), 3))
    plt.bar(keys, vals, color=[ORANGE if k in highlight else BLUE for k in keys])
    plt.xticks(rotation=90 if n >= 4 else 0); plt.ylabel("sayım"); plt.title(title, color=NAVY); plt.show()
print("hazır")

---
## A · Problem: "Bu girdi koşulu sağlıyor mu?"
Elimizde bir **kontrol fonksiyonu** `f(x)` var: `x` çözümse 1, değilse 0 döndürür. Fonksiyonun içini bilmiyoruz ya da işe yaramıyor (kara kutu). Klasik olarak yapabileceğimiz tek şey girdileri tek tek denemek: ortalama **N/2**, en kötü **N** çağrı.

Grover algoritması aynı işi **≈ (π/4)·√N** oracle çağrısıyla yapar. Hafta 1'deki grafiği hatırlayın: N = 1 milyon için 500.000 yerine ≈ 785 çağrı.

In [ ]:
N = 8
secret = 5                        # gizli çözüm (fonksiyonun içinde saklı)
calls = 0
def f(x):
    """Kontrol fonksiyonu (kara kutu): çözüm mü?"""
    global calls; calls += 1
    return int(x == secret)

for x in range(N):                # klasik arama: sırayla dene
    if f(x): break
print(f"Bulunan: {x} = {x:03b}, çağrı sayısı: {calls}")

# Ortalama çağrı sayısı (tüm olası gizli değerler için)
avg = np.mean([s + 1 for s in range(N)])
print(f"Klasik ortalama: {avg} ≈ N/2,   Grover: {int(np.floor(np.pi/4*np.sqrt(N)))} iterasyon")
for NN in [8, 1024, 10**6, 10**12]:
    print(f"N = {NN:>14,}: klasik ort. {NN/2:>16,.0f}   Grover ≈ {np.pi/4*np.sqrt(NN):>12,.0f}")

---
## B · NumPy ile Grover: iki basit işlem
Bir Grover iterasyonu, genlik vektörü üzerinde iki basit işlemden oluşur:
1. **Oracle:** hedefin genliğini −1 ile çarp (işaretini çevir). Olasılıklar değişmez! 8. haftadaki faz geri tepmesinin (phase kickback) aynısı.
2. **Difüzyon = ortalama etrafında yansıtma:** her genlik için `a → 2μ − a` (μ = genliklerin ortalaması).

Ortalamanın çok altındaki (negatif) hedef, yansıyınca ortalamanın çok üstüne fırlar.

In [ ]:
def np_oracle(a, marked):
    a = a.copy(); a[list(marked)] *= -1; return a

def np_diffusion(a):
    return 2*a.mean() - a

N, marked = 8, [5]
a = np.ones(N) / np.sqrt(N)
fig, axs = plt.subplots(1, 5, figsize=(20, 3.2))
plot_amps(a, marked, "Başlangıç", axs[0])
for it in range(2):
    a = np_oracle(a, marked);  plot_amps(a, marked, f"{it+1}. oracle", axs[1+2*it])
    a = np_diffusion(a);       plot_amps(a, marked, f"{it+1}. difüzyon  P = {a[5]**2:.3f}", axs[2+2*it])
plt.tight_layout(); plt.show()
print("2 iterasyon sonrası genlikler:", a)
print("P(hedef) =", a[5]**2)

**Elle kontrol (N = 8):** başlangıçta tüm genlikler 1/√8 ≈ 0.354.
- Oracle: hedef −0.354. Ortalama μ = (7·0.354 − 0.354)/8 = 0.265.
- Difüzyon: hedef 2·0.265 + 0.354 = **0.884** (P = 0.781), diğerleri 2·0.265 − 0.354 = 0.177.
- 2. iterasyonda hedef **0.972** (P = 0.945).

Difüzyonun matris hâli: **D = 2|s⟩⟨s| − I** (|s⟩ = eşit süperpozisyon). Kontrol edelim:

In [ ]:
s = np.ones(N) / np.sqrt(N)
D = 2*np.outer(s, s) - np.eye(N)
print("D köşegeni:", D[0, 0], "  köşegen dışı:", D[0, 1])
v = np.random.default_rng(1).normal(size=N)
print("D @ v == 2μ − v ?", np.allclose(D @ v, np_diffusion(v)))
print("D üniter mi?", np.allclose(D.T @ D, np.eye(N)))

---
## C · Qiskit: faz oracle ve difüzyon devreleri
**Oracle (faz):** Çok kontrollü Z (MCZ), yalnızca tüm kübitler 1 iken (|11…1⟩) işareti çevirir. Başka bir hedef için, hedefte **0 olan bitlere X** uygulayıp MCZ'den sonra geri alırız.

**Difüzyon:** `H⊗ⁿ · X⊗ⁿ · MCZ · X⊗ⁿ · H⊗ⁿ`. Ortadaki `X⊗ⁿ · MCZ · X⊗ⁿ` yalnızca |00…0⟩'ın işaretini çevirir; iki yandaki H⊗ⁿ bunu |s⟩ (eşit süperpozisyon) etrafında yansıtmaya dönüştürür. Sonuç 2|s⟩⟨s| − I'nın −1 katıdır (global faz). Global fazı `qc.global_phase = π` ile düzeltiyoruz ki genlikler NumPy ile birebir aynı çıksın.

In [ ]:
def mcz(qc, qubits):
    """Çok kontrollü Z: tüm kübitler 1 iken genliğin işaretini çevirir."""
    qubits = list(qubits)
    if len(qubits) == 1:   qc.z(qubits[0])
    elif len(qubits) == 2: qc.cz(qubits[0], qubits[1])
    else:                  qc.append(ZGate().control(len(qubits) - 1), qubits)

def oracle(n, marked):
    """Faz oracle: marked listesindeki bit dizilerinin ('101' gibi) işaretini çevirir."""
    if isinstance(marked, str): marked = [marked]
    qc = QuantumCircuit(n, name="Oracle")
    for s in marked:
        zeros = [i for i in range(n) if s[n-1-i] == "0"]   # Qiskit sırası: s[-1] = q0
        if zeros: qc.x(zeros)
        mcz(qc, range(n))
        if zeros: qc.x(zeros)
    return qc

def diffusion(n):
    """Difüzyon = 2|s⟩⟨s| − I  (ortalama etrafında yansıtma)."""
    qc = QuantumCircuit(n, name="Difüzyon")
    qc.h(range(n)); qc.x(range(n))
    mcz(qc, range(n))
    qc.x(range(n)); qc.h(range(n))
    qc.global_phase = np.pi
    return qc

display(oracle(3, "101").draw("mpl"))
display(diffusion(3).draw("mpl"))

In [ ]:
# Oracle'ı doğrula: eşit süperpozisyona uygula, sadece 101'in işareti dönmeli
qc = QuantumCircuit(3); qc.h(range(3)); qc.compose(oracle(3, "101"), inplace=True)
print(np.real(Statevector(qc).data))

# Difüzyonu doğrula: matrisi 2|s><s| − I ile aynı mı?
s = np.ones(8) / np.sqrt(8)
print("Difüzyon == 2|s⟩⟨s| − I ?", np.allclose(Operator(diffusion(3)).data, 2*np.outer(s, s) - np.eye(8)))

---
## D · Genel `grover(n, marked)` ve adım adım izleme
Optimum iterasyon sayısı: **k = ⌊(π/4)·√(N/M)⌋** (M = çözüm sayısı).

In [ ]:
def opt_iters(N, M=1):
    return int(np.floor(np.pi/4 * np.sqrt(N/M)))

def grover(n, marked, k=None, measure=True):
    """Genel Grover devresi. marked: '101' ya da ['010', '101']."""
    if isinstance(marked, str): marked = [marked]
    if k is None: k = opt_iters(2**n, len(marked))
    qc = QuantumCircuit(n, n) if measure else QuantumCircuit(n)
    qc.h(range(n))
    O, D = oracle(n, marked), diffusion(n)
    for _ in range(k):
        qc.barrier(); qc.compose(O, inplace=True); qc.compose(D, inplace=True)
    if measure:
        qc.barrier(); qc.measure(range(n), range(n))
    return qc

qc = grover(3, "101")
display(qc.draw("mpl", fold=-1))
counts = run_counts(qc)
print(counts)
plot_counts(counts, 3, ["101"], "n = 3, hedef 101, k = 2")

### Statevector ile adım adım genlik izleme
Devreyi ölçmeden, her oracle ve difüzyondan sonra `Statevector.evolve` ile durumu alıp çiziyoruz. Sonuç B bölümündeki NumPy hesabıyla birebir aynı olmalı.

In [ ]:
n, target = 3, "101"; idx = int(target, 2)
H = QuantumCircuit(n); H.h(range(n))
sv = Statevector.from_label("0"*n).evolve(H)
snapshots = [("Başlangıç (H⊗³)", sv)]
for it in range(1, 3):
    sv = sv.evolve(oracle(n, target));  snapshots.append((f"{it}. oracle", sv))
    sv = sv.evolve(diffusion(n));       snapshots.append((f"{it}. difüzyon", sv))

fig, axs = plt.subplots(1, 5, figsize=(20, 3.2))
for ax, (t, s_) in zip(axs, snapshots):
    plot_amps(s_.data, [idx], f"{t}\nP(101) = {abs(s_.data[idx])**2:.3f}", ax)
plt.tight_layout(); plt.show()

# NumPy ile karşılaştırma
a = np.ones(8)/np.sqrt(8)
for _ in range(2): a = np_diffusion(np_oracle(a, [idx]))
print("Statevector == NumPy ?", np.allclose(snapshots[-1][1].data, a))

In [ ]:
# n = 2, 3, 4 için devre boyutu ve başarı
for n in [2, 3, 4]:
    t = "1"*n; qc = grover(n, t)
    tq = transpile(qc, basis_gates=["cx", "u"], optimization_level=1, seed_transpiler=1)
    c = run_counts(qc)
    print(f"n={n}: k={opt_iters(2**n)}, derinlik={tq.depth():4d}, CX={tq.count_ops().get('cx',0):3d}, P({t})≈{c.get(t,0)/2000:.3f}")
display(grover(2, "11").draw("mpl"))

---
## E · Başarı olasılığı ve aşma (overshoot)
Geometrik görünüm: sin θ = √(M/N). Her iterasyon durumu **2θ** döndürür, k iterasyon sonra:
$$P(\text{hedef}) = \sin^2\big((2k+1)\theta\big)$$
Fazla iterasyon yaparsak hedefi **aşarız** ve olasılık yeniden düşer. Grover'da "ne kadar çok o kadar iyi" DEĞİLDİR.

In [ ]:
def success_prob_sv(n, marked, k):
    if isinstance(marked, str): marked = [marked]
    sv = Statevector(grover(n, marked, k, measure=False))
    return sum(abs(sv.data[int(m, 2)])**2 for m in marked)

plt.figure(figsize=(9, 3.8))
for n, c in [(3, ORANGE), (4, BLUE), (6, NAVY)]:
    N = 2**n; th = np.arcsin(np.sqrt(1/N)); ks = np.arange(0, 15)
    sim = [success_prob_sv(n, "1"*n, k) for k in ks]
    plt.plot(ks, np.sin((2*ks+1)*th)**2, "-", color=c, alpha=0.5)
    plt.plot(ks, sim, "o", color=c, label=f"N = {N} (optimum k = {opt_iters(N)})")
plt.xlabel("iterasyon k"); plt.ylabel("P(hedef)"); plt.legend(frameon=False); plt.grid(alpha=0.3)
plt.title("Noktalar: Statevector, çizgiler: sin²((2k+1)θ)", color=NAVY); plt.show()

---
## F · Birden fazla çözüm (M > 1)
Oracle birden fazla girdiyi işaretleyebilir. θ büyür → daha az iterasyon gerekir. Özel durum: **M = N/4** ise θ = 30° ve **tek iterasyonda P = 1**.

> **M bilinmiyorsa?** k'yi rastgele seçen artan bir strateji (Boyer–Brassard–Høyer–Tapp) ya da M'yi önce **kuantum sayma** (quantum counting, gelecek haftanın faz kestirimi ile) ile tahmin etmek gerekir.

In [ ]:
for M, marks in [(1, ["0101"]), (2, ["0101", "1100"]), (4, ["0001", "0101", "1010", "1111"])]:
    k = opt_iters(16, M); c = run_counts(grover(4, marks))
    p = sum(c.get(m, 0) for m in marks) / 2000
    print(f"N = 16, M = {M}: θ = {np.degrees(np.arcsin(np.sqrt(M/16))):5.2f}°, k = {k}, P(çözümlerden biri) ≈ {p:.3f}")
plot_counts(c, 4, marks, "N = 16, M = 4: tek iterasyon")

---
## G · Uygulama 1: 3 değişkenli SAT problemi
Formül (CNF): **F = (x₀ ∨ x₁) ∧ (¬x₀ ∨ x₂) ∧ (¬x₁ ∨ ¬x₂)**

Oracle'ı "çözümü bilerek" değil, **formülün kendisinden** kuruyoruz — gerçek kullanım budur:
1. Her cümle (clause) için bir **yardımcı kübit**: `(a ∨ b) = ¬(¬a ∧ ¬b)` → X'ler + Toffoli + X.
2. Tüm cümle kübitleri 1 ise **çıkış kübitini** çevir (çıkış |−⟩ hâlinde → faz geri tepmesi, Hafta 8).
3. **Uncompute:** cümle hesaplarını ters sırada geri al, yardımcılar yeniden |0⟩ olsun. Yoksa çöp (garbage) kübitler dolanık kalır ve girişim bozulur.

In [ ]:
CLAUSES = [[(0, True), (1, True)], [(0, False), (2, True)], [(1, False), (2, False)]]   # (değişken, pozitif mi?)

def clause_gate(qc, clause, anc):
    """anc <- (lit1 ∨ lit2).  (a ∨ b) = NOT(¬a ∧ ¬b)."""
    flip = [v for v, pos in clause if pos]          # pozitif literalleri çevir → ¬a
    if flip: qc.x(flip)
    qc.ccx(clause[0][0], clause[1][0], anc)          # anc = ¬a ∧ ¬b
    if flip: qc.x(flip)
    qc.x(anc)                                        # anc = a ∨ b

def sat_oracle(clauses, n_vars):
    m = len(clauses); out = n_vars + m
    qc = QuantumCircuit(n_vars + m + 1, name="SAT")
    for j, cl in enumerate(clauses): clause_gate(qc, cl, n_vars + j)       # hesapla
    qc.mcx(list(range(n_vars, n_vars + m)), out)                          # hepsi 1 ise çıkışı çevir
    for j, cl in reversed(list(enumerate(clauses))): clause_gate(qc, cl, n_vars + j)   # uncompute
    return qc

def sat_grover(clauses, n_vars, k):
    m = len(clauses); out = n_vars + m
    qc = QuantumCircuit(n_vars + m + 1, n_vars)
    qc.x(out); qc.h(out)                              # çıkış kübiti |−⟩
    qc.h(range(n_vars))
    for _ in range(k):
        qc.barrier(); qc.compose(sat_oracle(clauses, n_vars), inplace=True)
        qc.compose(diffusion(n_vars), range(n_vars), inplace=True)
    qc.barrier(); qc.measure(range(n_vars), range(n_vars))
    return qc

display(sat_oracle(CLAUSES, 3).draw("mpl", fold=-1))

# Klasik kontrol: çözümler
def check(bits, clauses):
    x = [(bits >> i) & 1 for i in range(3)]
    return all(any(x[v] == int(p) for v, p in cl) for cl in clauses)
sols = [format(b, "03b") for b in range(8) if check(b, CLAUSES)]
print("Klasik çözümler (x2x1x0):", sols)

c = run_counts(sat_grover(CLAUSES, 3, k=opt_iters(8, len(sols))))
print(c); plot_counts(c, 3, sols, f"SAT: M = {len(sols)}, k = {opt_iters(8, len(sols))}")

In [ ]:
# Uncompute kontrolü: oracle'dan sonra yardımcı kübitler |0⟩'a dönmeli
qc = QuantumCircuit(7); qc.x(6); qc.h(6); qc.h(range(3)); qc.compose(sat_oracle(CLAUSES, 3), inplace=True)
probs = Statevector(qc).probabilities_dict(qargs=[3, 4, 5])
print("Yardımcı kübitlerin (q5q4q3) dağılımı:", probs)

---
## H · Uygulama 2: 4 bitlik şifre bulma ve listede arama
Bir "kilit" düşünün: `check(p)` fonksiyonu şifre doğruysa 1 döndürür. 4 bit → 16 olasılık. Klasik ortalama 8 deneme, Grover **3 iterasyon**.

⚠️ Gerçekçi not: Burada oracle'ı şifreyi bilerek kuruyoruz (ders amaçlı). Gerçek bir uygulamada oracle, `check` fonksiyonunun **tersinir devre** hâli olurdu (ör. bir özet/hash fonksiyonu) ve bu devre çok büyük olurdu.

In [ ]:
secret = "1011"
qc = grover(4, secret)
c = run_counts(qc)
best = max(c, key=c.get)
print(f"En sık sonuç: {best}  (P ≈ {c[best]/2000:.3f}),  gerçek şifre: {secret}")
plot_counts(c, 4, [secret], "4 bitlik şifre, k = 3")

In [ ]:
# Listede öğe bulma: 8 isimden "Ayşe" hangi indekste?
names = ["Ali", "Ece", "Can", "Deniz", "Selin", "Ayşe", "Mert", "Zeynep"]
marked = [format(i, "03b") for i, nm in enumerate(names) if nm == "Ayşe"]   # oracle'ı klasik olarak kurduk!
c = run_counts(grover(3, marked))
best = max(c, key=c.get)
print("Bulunan indeks:", best, "=", int(best, 2), "->", names[int(best, 2)])
print("Dikkat: oracle'ı kurmak için listeyi zaten taradık (O(N)). Veri yükleme maliyeti avantajı yok eder.")

---
## I · Gerçekçi değerlendirme: gürültü
Grover devresinin derinliği n ile hızla büyür (MCZ ayrıştırması + √N iterasyon). Basit bir depolarizing gürültü modeliyle başarının nasıl düştüğüne bakalım.

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error
def noise(p2):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p2/10, 1), ["u"])
    nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ["cx"])
    return nm
for p2 in [0.0, 0.005, 0.02]:
    sim = AerSimulator(noise_model=noise(p2), basis_gates=["cx", "u"], seed_simulator=3) if p2 else AerSimulator(seed_simulator=3)
    row = []
    for n in [2, 3, 4, 5]:
        tq = transpile(grover(n, "1"*n), basis_gates=["cx", "u"], optimization_level=1, seed_transpiler=1)
        row.append(sim.run(tq, shots=2000).result().get_counts().get("1"*n, 0) / 2000)
    print(f"CX hatası {p2:5.3f}:  " + "  ".join(f"n={n}: {p:.2f}" for n, p in zip([2, 3, 4, 5], row)))
print("Rastgele tahmin 1/N:  " + "  ".join(f"n={n}: {1/2**n:.2f}" for n in [2, 3, 4, 5]))

---
## J · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur.

### Alıştırma 1 · Optimum iterasyon sayısı
`optimal_iterations(N, M)` = ⌊(π/4)·√(N/M)⌋ döndürsün.

In [ ]:
def optimal_iterations(N, M=1):
    # TODO
    pass

assert optimal_iterations(4) == 1
assert optimal_iterations(8) == 2
assert optimal_iterations(64) == 6
assert optimal_iterations(16, 4) == 1
assert optimal_iterations(2**20) == 804
print("Alıştırma 1 ✓")

### Alıştırma 2 · Difüzyon matrisi
`diffusion_matrix(N)` = 2|s⟩⟨s| − I matrisini döndürsün. Ardından N = 4, hedef 3 (|11⟩) için NumPy ile bir iterasyon uygulayın ve P = 1 olduğunu gösterin.

In [ ]:
def diffusion_matrix(N):
    # TODO
    pass

D4 = diffusion_matrix(4)
a = np.ones(4) / 2
a = D4 @ np_oracle(a, [3])
print(a)
assert np.allclose(D4, D4.T) and np.allclose(D4 @ D4, np.eye(4))   # yansıma: kendi tersi
assert np.isclose(abs(a[3])**2, 1.0)
assert np.allclose(diffusion_matrix(8) @ np.arange(8.0), np_diffusion(np.arange(8.0)))
print("Alıştırma 2 ✓")

### Alıştırma 3 · Kendi faz oracle'ınız
`my_oracle(n, target)`: yalnızca `target` bit dizisinin işaretini çeviren devre. (İpucu: 0 olan bitlere X, `mcz`, tekrar X. Qiskit sırasına dikkat: `target[-1]` q₀'dır.) Operatörün köşegen olduğunu ve yalnızca hedefte −1 bulunduğunu kontrol ediyoruz.

In [ ]:
def my_oracle(n, target):
    qc = QuantumCircuit(n)
    # TODO
    return qc

for n, t in [(2, "10"), (3, "011"), (4, "0110")]:
    U = Operator(my_oracle(n, t)).data
    expected = np.eye(2**n); expected[int(t, 2), int(t, 2)] = -1
    assert np.allclose(U, expected), (n, t)
print("Alıştırma 3 ✓")

### Alıştırma 4 · Kendi difüzyon devreniz
`my_diffusion(n)` devresini H, X ve `mcz` ile kurun. `Operator.equiv` global fazı yok sayarak karşılaştırır.

In [ ]:
def my_diffusion(n):
    qc = QuantumCircuit(n)
    # TODO
    return qc

for n in [2, 3, 4]:
    s = np.ones(2**n) / np.sqrt(2**n)
    assert Operator(my_diffusion(n)).equiv(Operator(2*np.outer(s, s) - np.eye(2**n))), n
print("Alıştırma 4 ✓")

### Alıştırma 5 · Başarı eğrisi ve aşma
`success_curve(n, target, kmax)`: k = 0..kmax için Statevector ile P(hedef) listesini döndürsün. N = 16 için en iyi k'yi bulun ve formülle karşılaştırın.

In [ ]:
def success_curve(n, target, kmax):
    # TODO: her k için grover(n, target, k, measure=False) devresinin Statevector'ından P(hedef)
    pass

curve = success_curve(4, "0110", 8)
th = np.arcsin(1/4)
assert np.allclose(curve, [np.sin((2*k+1)*th)**2 for k in range(9)], atol=1e-6)
assert int(np.argmax(curve)) == 3 and curve[3] > 0.96
assert curve[6] < 0.05          # aşma: 6 iterasyonda neredeyse sıfır
print(np.round(curve, 3)); print("Alıştırma 5 ✓")

### Alıştırma 6 · Birden fazla çözüm
N = 32 (n = 5), M = 3 çözüm: `['00011', '01100', '11110']`. Optimum k'yi hesaplayın, Statevector ile toplam başarı olasılığını bulun.

In [ ]:
marks = ["00011", "01100", "11110"]
k6 = None        # TODO
p6 = None        # TODO: Statevector ile üç çözümün olasılıkları toplamı

print(k6, p6)
assert k6 == 2
assert p6 > 0.99
print("Alıştırma 6 ✓")

### Alıştırma 7 · Yeni SAT formülü
**G = (x₀ ∨ ¬x₁) ∧ (x₁ ∨ x₂) ∧ (¬x₀ ∨ ¬x₂)**. (a) Cümle listesini yazın. (b) Çözümleri klasik olarak bulun (`check` fonksiyonu). (c) `sat_grover` ile çalıştırın; en sık M sonucun klasik çözümlerle aynı olduğunu gösterin.

In [ ]:
G_CLAUSES = None       # TODO: [[(değişken, pozitif_mi), (değişken, pozitif_mi)], ...]
g_sols = None          # TODO: klasik çözümler, x2x1x0 bit dizileri
counts7 = None         # TODO: sat_grover(...) sayımları (2000 shot)

top = sorted(counts7, key=counts7.get, reverse=True)[:len(g_sols)]
print(g_sols, counts7)
assert sorted(g_sols) == ["011", "100"]
assert sorted(top) == sorted(g_sols)
assert sum(counts7[s] for s in g_sols) / 2000 > 0.95
print("Alıştırma 7 ✓")

### Alıştırma 8 · 5 bitlik şifre kırma
Gizli şifre `'10110'`. `crack(secret)` fonksiyonu: Grover devresini kurup çalıştırsın ve `(en_sik_sonuc, oracle_cagri_sayisi)` döndürsün. Klasik ortalama çağrı sayısı ile karşılaştırın.

In [ ]:
def crack(secret, shots=2000):
    n = len(secret)
    # TODO
    pass

guess, calls = crack("10110")
print("tahmin:", guess, " oracle çağrısı:", calls, " klasik ortalama:", 2**5 / 2)
assert guess == "10110"
assert calls == 4
assert calls < 2**5 / 2
print("Alıştırma 8 ✓")

---
### Haftanın özeti
- Sırasız arama: klasik **O(N)**, Grover **O(√N)** oracle çağrısı (karesel hızlanma, üstel değil)
- Adımlar: **H⊗ⁿ** (eşit süperpozisyon) → [**oracle** (işaret çevir) → **difüzyon** (ortalama etrafında yansıt)] × k → ölçüm
- Geometri: sin θ = √(M/N), her iterasyon **2θ** dönüş; P = sin²((2k+1)θ); optimum **k = ⌊(π/4)√(N/M)⌋**; fazlası **aşma**
- Devre: oracle = X + MCZ + X; difüzyon = H X MCZ X H; derinlik n ile hızla artar
- Uygulamada oracle'ı **problemin kendisinden** kurarız (SAT: yardımcı kübit + uncompute)
- Gerçekçi bakış: oracle maliyeti, veri yükleme ve gürültü avantajı küçük n'de siler

**Gelecek hafta:** Kuantum Fourier Dönüşümü (QFT), faz kestirimi (QPE) ve Shor algoritması ile N = 15'i çarpanlarına ayırma. Faz kestirimi, bu haftaki "M bilinmiyorsa" sorusunun da (kuantum sayma) cevabıdır.